# Task 1: The Accuracy Metric Validation

In this section, we implement the `calculate_accuracy` function. 

**Consultant Insight:** While **Loss** is calculated using Binary Cross-Entropy (which is essential for creating a smooth gradient during backpropagation), **Accuracy** requires a hard threshold. For a binary classification with a Sigmoid output, we round the probabilistic output (e.g., $p \ge 0.5 \rightarrow 1$, else $0$) and compare it to the target labels.

In [1]:
import numpy as np

def calculate_accuracy(y_pred: np.ndarray, y_true: np.ndarray) -> float:
    """
    Calculates the accuracy of the model's predictions compared to the true labels.
    
    Why: 
    While BCE Loss gives us a continuous gradient for backpropagation, human 
    interpretation requires a concrete metric. Since our final layer uses a 
    Sigmoid activation, the output is a probability between 0 and 1. We apply 
    a threshold (0.5) to convert these probabilities into binary classifications 
    (0 for Benign, 1 for Malignant).
    """
    # 1. Apply threshold: Convert probabilities to binary predictions
    predictions = (y_pred >= 0.5).astype(int)
    
    # 2. Compare to true labels and calculate the mean of correct matches
    accuracy = np.mean(predictions == y_true)
    
    return float(accuracy)

In [2]:
# --- Validation Test ---
y_true_dummy = np.array([[1], [0], [1], [1]])
y_pred_dummy = np.array([[0.8], [0.4], [0.45], [0.9]])

acc = calculate_accuracy(y_pred_dummy, y_true_dummy)
print(f"Raw Probabilities:\n{y_pred_dummy.flatten()}")
print(f"Binary Predictions: {(y_pred_dummy >= 0.5).astype(int).flatten()}")
print(f"True Labels:        {y_true_dummy.flatten()}")
print(f"\n-> Calculated Accuracy: {acc * 100}%")

Raw Probabilities:
[0.8  0.4  0.45 0.9 ]
Binary Predictions: [1 0 0 1]
True Labels:        [1 0 1 1]

-> Calculated Accuracy: 75.0%


# Task 2: Define `update_parameters` and Parameter Math

### Mathematics of Gradient Descent

The goal of backpropagation is to calculate $dW$ and $db$, which represent the gradients (derivatives) of the Loss function with respect to the Weights and Biases. 
In other words, they tell us the *direction* and *steepness* of the error curve.

To minimize the error, we update our parameters by taking a small step in the **opposite direction** of the gradient. 

The update formulas are:
$$W = W - (\text{learning\_rate} \cdot dW)$$
$$b = b - (\text{learning\_rate} \cdot db)$$

**The Role of the Learning Rate ($\alpha$):**
- **Too large:** The steps are massive, and we might "overshoot" the minimum error, causing the loss to explode or bounce around erratically.
- **Too small:** The steps are tiny, meaning training will take forever and might get stuck in shallow local minima.

### Refactoring Strategy
Currently, `layer.py` applies `self.weights -= learning_rate * dW` directly inside its `backward` method. To adhere to standard practices and support batching, we will:
1. Ensure `backward` just stores `self.dW` and `self.db`.
2. Let the main `MultilayerPerceptron` class control when things update via an `update_parameters` method.

In [3]:
def update_parameters(network_layers, learning_rate: float):
    """
    Iterates through all layers in the network and applies the gradient descent 
    update rule to modify weights and biases.
    
    Args:
        network_layers: List of layer objects (e.g., DenseLayer instances).
        learning_rate: The step size multiplier for the gradients.
    """
    for layer in network_layers:
        # Assuming the layer's backward method has already saved dW and db as attributes.
        # W = W - (alpha * dW)
        layer.weights -= learning_rate * layer.dW
        # b = b - (alpha * db)
        layer.biases -= learning_rate * layer.db

# --- Prototype Validation ---
# Let's mock a layer object to verify the math
class MockLayer:
    def __init__(self):
        self.weights = np.array([[0.5, -0.2]])
        self.biases = np.array([[0.0, 0.0]])
        # Mock calculating gradients during backward pass
        self.dW = np.array([[0.1, -0.4]])
        self.db = np.array([[0.05, -0.1]])

layers = [MockLayer()]
learning_rate = 0.1

print(f"Original Weights:\\n{layers[0].weights}")
print(f"Original Biases:\\n{layers[0].biases}\\n")

update_parameters(layers, learning_rate)

print(f"Updated Weights:\\n{layers[0].weights}")
print(f"Updated Biases:\\n{layers[0].biases}")


Original Weights:\n[[ 0.5 -0.2]]
Original Biases:\n[[0. 0.]]\n
Updated Weights:\n[[ 0.49 -0.16]]
Updated Biases:\n[[-0.005  0.01 ]]


# Task 3: Full Epoch Training Loop Prototype

We will now stitch the forward pass, loss calculation, backward pass, and parameter update into the complete "Epoch" loop.  

An epoch is a single iteration over the entirety of the training dataset. At the end of each epoch, we will calculate our metrics on both the training set and the validation set.

*Note: For this prototype we will use dummy data and mock the forward/backward pass, so we can clearly understand the orchestration before implementing it inside the `MultilayerPerceptron` class.*

In [ ]:
import sys
import os
sys.path.append('..')

from src.loss import binary_cross_entropy, binary_cross_entropy_prime

# Let's mock a network for the training loop
class MockNetwork:
    def __init__(self):
        # We will mock the output so the loss changes
        self.epoch = 0
            
    def forward(self, X):
        # Mocking forward pass: prediction gets closer to y_true as epochs increase
        # Return an array matching the input batch size
        # [:n] means we take only the first n rows to match the input batch size(the shape of X)
        n = X.shape[0]
        if self.epoch == 0:
            return np.array([[0.1], [0.9], [0.1], [0.8]])[:n]
        elif self.epoch == 1:
            return np.array([[0.3], [0.7], [0.2], [0.85]])[:n]
        else:
            return np.array([[0.9], [0.1], [0.9], [0.9]])[:n]
            
    def backward(self, loss_grad):
        # Mock backward pass: simply takes the gradient
        pass
        
    def update_parameters(self, learning_rate):
        # Mock weight updates
        self.epoch += 1


In [7]:

# Dummy Data
X_train = np.array([[0,0], [1,1], [0,1], [1,0]])
y_train = np.array([[1], [0], [1], [1]])

X_val = np.array([[0,0], [1,1]])
y_val = np.array([[1], [0]])

# Hyperparameters
epochs = 3
learning_rate = 0.01

# Initialize our mocked network
model = MockNetwork()
history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}


In [8]:
print("Starting Training Loop...")
for epoch in range(epochs):
    # 1. Forward Pass (Training)
    y_pred_train = model.forward(X_train)

    # 2. Compute Loss and Accuracy (Training)
    train_loss = binary_cross_entropy(y_train, y_pred_train)
    train_acc = calculate_accuracy(y_pred_train, y_train)

    # 3. Backward Pass (Training)
    # Start the chain reaction by computing the gradient of the loss at the output
    loss_gradient = binary_cross_entropy_prime(y_train, y_pred_train)
    model.backward(loss_gradient)

    # 4. Update Weights
    model.update_parameters(learning_rate)

    # 5. Validation Pass (End of Epoch)
    # We do NOT run backward or update on validation data!
    y_pred_val = model.forward(
        X_val
    )  # uses updated weights because update_parameters was called
    val_loss = binary_cross_entropy(y_val, y_pred_val)
    val_acc = calculate_accuracy(y_pred_val, y_val)

    # 6. Store metrics for history
    history["loss"].append(train_loss)
    history["accuracy"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_acc)

    # 7. Print Output (Task 4 preview)
    print(
        f"Epoch {epoch+1:02d}/{epochs} - loss: {train_loss:.4f} - accuracy: {train_acc:.4f} - val_loss: {val_loss:.4f} - val_acc: {val_acc:.4f}"
    )

Starting Training Loop...
Epoch 01/3 - loss: 1.7827 - accuracy: 0.2500 - val_loss: 1.2040 - val_acc: 0.0000
Epoch 02/3 - loss: 1.0450 - accuracy: 0.2500 - val_loss: 0.1054 - val_acc: 1.0000
Epoch 03/3 - loss: 0.1054 - accuracy: 1.0000 - val_loss: 0.1054 - val_acc: 1.0000


# Production implementation


In [5]:

"""
Training Program

This script:
1) initializes the Multilayer Perceptron
2) runs the training loop using feedforward and backpropagation
3) plots the learning curves (loss and accuracy)
and persists the trained topology and weights to the models/ directory.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
from src.network import MultilayerPerceptron
from src.scaler import StandardScaler, encode_labels
from src.loss import binary_cross_entropy, binary_cross_entropy_prime


In [6]:

"""
Executes the training phase on the partitioned training data.

Why: This is the core learning loop. It coordinates data loading, scaling,
forward passes, error calculation (BCE), and the crucial backpropagation
step where weights are updated via Gradient Descent.
"""

'\nExecutes the training phase on the partitioned training data.\n\nWhy: This is the core learning loop. It coordinates data loading, scaling,\nforward passes, error calculation (BCE), and the crucial backpropagation\nstep where weights are updated via Gradient Descent.\n'

## Load data


In [7]:

# 1. Load Data
try:
    train_df = pd.read_csv("../data/training_data.csv", header=None)
    val_df = pd.read_csv("../data/validation_data.csv", header=None)
except FileNotFoundError:
    print("Error: Partitioned data not found. Please run split.py first.")


In [17]:
train_df.shape, val_df.shape


((456, 32), (113, 32))

In [18]:

train_df.head()


,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,88995002,M,20.730,31.12,135.70,1419.0,0.09469,0.11430,0.13670,0.08646,...,32.49,47.16,214.00,3432.0,0.14010,0.2644,0.3442,0.16590,0.2868,0.08218
1,859471,B,9.029,17.33,58.79,250.5,0.10660,0.14130,0.31300,0.04375,...,10.31,22.65,65.50,324.7,0.14820,0.4365,1.2520,0.17500,0.4228,0.11750
2,873593,M,21.090,26.57,142.70,1311.0,0.11410,0.28320,0.24870,0.14960,...,26.68,33.48,176.50,2089.0,0.14910,0.7584,0.6780,0.29030,0.4098,0.12840
3,859196,B,9.173,13.86,59.20,260.9,0.07721,0.08751,0.05988,0.02180,...,10.01,19.23,65.59,310.1,0.09836,0.1678,0.1397,0.05087,0.3282,0.08490
4,88466802,B,10.650,25.22,68.01,347.0,0.09657,0.07234,0.02379,0.01615,...,12.25,35.19,77.98,455.7,0.14990,0.1398,0.1125,0.06136,0.3409,0.08147


In [19]:

val_df.head()

,0,1,2,3,4,5,6,7,8,9,...,22,23,24,25,26,27,28,29,30,31
0,87930,B,12.47,18.60,81.09,481.9,0.09965,0.1058,0.08005,0.03821,...,14.97,24.64,96.05,677.9,0.1426,0.2378,0.2671,0.10150,0.3014,0.08750
1,859575,M,18.94,21.31,123.60,1130.0,0.09009,0.1029,0.10800,0.07951,...,24.86,26.58,165.90,1866.0,0.1193,0.2336,0.2687,0.17890,0.2551,0.06589
2,8670,M,15.46,19.48,101.70,748.9,0.10920,0.1223,0.14660,0.08087,...,19.26,26.00,124.90,1156.0,0.1546,0.2394,0.3791,0.15140,0.2837,0.08019
3,907915,B,12.40,17.68,81.47,467.8,0.10540,0.1316,0.07741,0.02799,...,12.88,22.91,89.61,515.8,0.1450,0.2629,0.2403,0.07370,0.2556,0.09359
4,921385,B,11.54,14.44,74.65,402.9,0.09984,0.1120,0.06737,0.02594,...,12.26,19.68,78.78,457.8,0.1345,0.2118,0.1797,0.06918,0.2329,0.08134


In [10]:
# Slice data: Col 0 is ID, Col 1 is Label, Col 2-31 are features
#! This in order to separted feature from the target for (training and validation data)
X_train_raw = train_df.iloc[:, 2:].values
y_train_raw = train_df.iloc[:, 1].values
X_val_raw = val_df.iloc[:, 2:].values
y_val_raw = val_df.iloc[:, 1].values

In [11]:
X_train_raw[10,:]

array([1.989e+01, 2.026e+01, 1.305e+02, 1.214e+03, 1.037e-01, 1.310e-01,
       1.411e-01, 9.431e-02, 1.802e-01, 6.188e-02, 5.079e-01, 8.737e-01,
       3.654e+00, 5.970e+01, 5.089e-03, 2.303e-02, 3.052e-02, 1.178e-02,
       1.057e-02, 3.391e-03, 2.373e+01, 2.523e+01, 1.605e+02, 1.646e+03,
       1.417e-01, 3.309e-01, 4.185e-01, 1.613e-01, 2.549e-01, 9.136e-02])

In [12]:

y_train_raw

<StringArray>
['M', 'B', 'M', 'B', 'B', 'B', 'M', 'B', 'B', 'B',
 ...
 'M', 'M', 'B', 'M', 'B', 'B', 'B', 'B', 'M', 'B']
Length: 456, dtype: str

## Processing data

In [13]:
# 2. Preprocessing
# Standardize features (Mean=0, Std=1) to ensure smooth gradient descent convergence.
# We call the object to standardize all the data
scaler = StandardScaler()

# Calculates and save parameters (mean and std) exclusively on training data to prevent data leakage
scaler.fit(X_train_raw)

# We standardize the training data to use the same scaling mesurement everywhere
X_train = scaler.transform(X_train_raw)
# We standardize iqually the validation data so we use the same scale everywhere
X_val = scaler.transform(X_val_raw)

# we save the scales data
scaler.save("models/scaler.json")

In [14]:
# Encode labels (M=1, B=0) for mathematical compatibility with BCE loss
#reshape(-1, 1) is to make sure the labels are in the correct shape (n_samples, 1) for matrix operations in the network
y_train = encode_labels(y_train_raw).reshape(-1, 1)


In [20]:
type(y_train)

numpy.ndarray

In [ ]:
# A column with 456 rows
y_train.shape

(456, 1)

In [ ]:
# Example data after encoding and reshaping
y_train.tolist()[:10]

[[1], [0], [1], [0], [0], [0], [1], [0], [0], [0]]

In [ ]:

# Hot encoding the validation labels as well -1 means all the rows and 1 means one column
y_val = encode_labels(y_val_raw).reshape(-1, 1)

## 3. Initialize Network

In [25]:

# Topology: [Input (30 features), Hidden1 (32), Hidden2 (32), Output (1)]
# We use two hidden layers to satisfy architectural requirements.
topology = [X_train.shape[1], 32, 32, 1]

mlp = MultilayerPerceptron(
    topology, hidden_activation="relu", output_activation="sigmoid"
)
mlp.summary()

Layer (Type)         Shape (In, Out)      Param #        
Dense-1 (relu)       (30, 32)             992            
------------------------------------------------------------
Dense-2 (relu)       (32, 32)             1056           
------------------------------------------------------------
Dense-3 (sigmoid)    (32, 1)              33             
------------------------------------------------------------
Total params: 2081


## 4. Hyperparameters




In [28]:

epochs = 1000
learning_rate = 0.1
history = {"loss": [], "val_loss": [], "acc": [], "val_acc": []}

print(f"\nStarting training for {epochs} epochs...")



Starting training for 1000 epochs...


In [ ]:



# 5. Training Loop
for epoch in range(epochs):
    # --- Forward Pass ---
    y_pred_train = mlp.forward(X_train)
    train_loss = binary_cross_entropy(y_train, y_pred_train)

    # --- Backward Pass & Task 5 (Weight Updates) ---
    # We calculate the initial gradient of the loss with respect to the output.
    loss_grad = binary_cross_entropy_prime(y_train, y_pred_train)
    # mlp.backward propagates this signal and updates weights via Gradient Descent.
    mlp.backward(loss_grad, learning_rate)

    # --- Validation & Metrics ---
    y_pred_val = mlp.forward(X_val)
    val_loss = binary_cross_entropy(y_val, y_pred_val)

    # Calculate accuracy for tracking
    train_acc = np.mean((y_pred_train > 0.5) == y_train)
    val_acc = np.mean((y_pred_val > 0.5) == y_val)

    # Log history
    history["loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    if epoch % 100 == 0 or epoch == epochs - 1:
        print(
            f"Epoch {epoch:4d}/{epochs} | Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
        )

In [ ]:

# 6. Finalization
os.makedirs("models", exist_ok=True)
mlp.save_model("models/mlp_model.json")
plot_learning_curves(history)